# Convert Indoor Segmentation Model to Core ML

This notebook converts the PyTorch model to Core ML format.

## Steps:
1. Upload `IndoorSegmentation.pt`
2. Run all cells
3. Download `IndoorSegmentation.mlpackage`

In [ ]:
# Install dependencies
!pip install -q torch coremltools

In [ ]:
# Upload your IndoorSegmentation.pt file
from google.colab import files
uploaded = files.upload()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import coremltools as ct

class PointNetSegmentation(nn.Module):
    def __init__(self, num_classes=4, input_channels=6):
        super().__init__()
        self.enc1 = nn.Conv1d(input_channels, 64, 1)
        self.enc2 = nn.Conv1d(64, 128, 1)
        self.enc3 = nn.Conv1d(128, 256, 1)
        self.enc4 = nn.Conv1d(256, 512, 1)
        self.enc5 = nn.Conv1d(512, 1024, 1)
        self.dec1 = nn.Conv1d(1024 + 64, 512, 1)
        self.dec2 = nn.Conv1d(512, 256, 1)
        self.dec3 = nn.Conv1d(256, 128, 1)
        self.dec4 = nn.Conv1d(128, num_classes, 1)

    def forward(self, x):
        num_points = x.size(1)
        x = x.transpose(2, 1)
        x1 = F.relu(self.enc1(x))
        x2 = F.relu(self.enc2(x1))
        x3 = F.relu(self.enc3(x2))
        x4 = F.relu(self.enc4(x3))
        x5 = F.relu(self.enc5(x4))
        global_feat = torch.max(x5, 2, keepdim=True)[0]
        global_feat = global_feat.repeat(1, 1, num_points)
        x = torch.cat([global_feat, x1], dim=1)
        x = F.relu(self.dec1(x))
        x = F.relu(self.dec2(x))
        x = F.relu(self.dec3(x))
        x = self.dec4(x)
        x = x.transpose(2, 1)
        x = F.softmax(x, dim=-1)
        return x

print("Model class defined!")

In [ ]:
# Load checkpoint
checkpoint = torch.load("IndoorSegmentation.pt", map_location="cpu")
model = PointNetSegmentation(num_classes=4, input_channels=6)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Model loaded!")

In [ ]:
# Trace and convert
example_input = torch.randn(1, 4096, 6)
traced = torch.jit.trace(model, example_input)

mlmodel = ct.convert(
    traced,
    inputs=[ct.TensorType(name="points", shape=(1, ct.RangeDim(100, 100000, 4096), 6), dtype=np.float32)],
    outputs=[ct.TensorType(name="classifications", dtype=np.float32)],
    minimum_deployment_target=ct.target.iOS15,
    compute_precision=ct.precision.FLOAT16,
)

mlmodel.author = "LiDAR Scanner"
mlmodel.short_description = "Indoor scene semantic segmentation"
mlmodel.save("IndoorSegmentation.mlpackage")
print("Core ML model saved!")

In [ ]:
# Download the model
!zip -r IndoorSegmentation.mlpackage.zip IndoorSegmentation.mlpackage
files.download("IndoorSegmentation.mlpackage.zip")